# IFA2 Interconnector — Spread-Direct OU Model (Methodology 2)
**MSc Climate Change Finance and Investment**

## Background: Why a Second Methodology?

Methodology 1 (`IC_descriptive_stats.ipynb`) follows Abadie & Chamorro (2021): model `log(P_GB)` and `log(P_FR)` as separate OU+jump processes. Revenue = |exp(f_GB + X_GB) − exp(f_FR + X_FR)|.

This produced **P50 revenues of £305m/yr** against Ofgem-disclosed IFA2 actuals of **£108–188m/yr**.

**Root cause — Jensen's inequality:**

> E[|exp(X_GB) − exp(X_FR)|] >> |exp(E[X_GB]) − exp(E[X_FR])| when σ is large

Our log-price residual volatilities (σ_GB = 0.240, σ_FR = 0.421 per day) are driven up by the 2021–2022 energy crisis. Abadie's Spain-France data predates that crisis (σ ≈ 0.05), making the inflation factor exp(σ²/2) ≈ 1.001 — negligible. Our high-volatility regime breaks the approximation.

## This Notebook: Model the Spread Directly

Model S_t = P_GB_t − P_FR_t (£/MWh, daily, in levels) as OU + jumps:

    S_t = f_S(t) + X_t
    X_{t+1} = c + φ·X_t + σ_d·ε_t + J_t

No log transformation → no Jensen's inequality inflation. Follows Cartea & González-Pedraz (2012).

**Pipeline:** Data Load → Clean → Descriptive Stats → EDA Plots → Full OLS → Significance Tests → Refined OLS → Jump Detection → OU Estimation → Monte Carlo → Results → Excel Export


## Section 1: Configuration & Imports

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats as sp_stats
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True,
    'grid.linewidth': 0.3, 'grid.alpha': 0.5, 'font.size': 10,
})

IC_DIR     = Path('/Users/aadesh/Documents/IC')
EXCEL_PATH = IC_DIR / 'Cleaned_GBFR.xlsx'
OUTPUT_DIR = IC_DIR / 'M2'
OUTPUT_DIR.mkdir(exist_ok=True)

FID_YEAR        = 2028       # First revenue year (IFA2 operational)
REGIME_YEARS    = 25         # Cap & Floor regime duration
CAPACITY_MW     = 1_000      # Nameplate capacity (MW)
AVAILABILITY    = 0.9659     # Technical availability (Ofgem IFA2 disclosure)
N_PATHS         = 10_000     # Monte Carlo paths
RANDOM_SEED     = 42
PROJ_TAU_GROWTH = 0.0        # 0 = flat nominal base case
ANCHOR_DATE     = pd.Timestamp('2025-01-01')  # tau frozen at mid 2024-2025 period

# Ofgem-disclosed IFA2 gross congestion revenues net of market costs (£m)
ACTUALS = {2022: 134.9, 2023: 188.3, 2024: 107.5, 2025: 109.4}

print(f'Config: FID={FID_YEAR}  REGIME={REGIME_YEARS}yr  '
      f'CAPACITY={CAPACITY_MW}MW  AVAILABILITY={AVAILABILITY}')
print(f'        N_PATHS={N_PATHS:,}  SEED={RANDOM_SEED}  '
      f'ANCHOR={ANCHOR_DATE.date()}')


## Section 2: Data Loading

Data source: `Cleaned_GBFR.xlsx` — Bloomberg GB and FR hourly electricity prices + EUR/GBP FX.

| Sheet | Content | Units |
|---|---|---|
| `GB_FINAL` | N2EX GB day-ahead hourly prices | GBP/MWh |
| `FR_FINAL` | EPEX France day-ahead hourly prices | EUR/MWh |
| `GBP-EUR` | Bloomberg EUR/GBP daily FX rate | GBP per 1 EUR |

**Wide-to-long:** Each sheet stores dates as rows, hours (H01–H24) as columns. `_wide_to_hourly()` pivots to a timestamp-indexed Series.

**FX convention:** FR_price_GBP = FR_price_EUR × fx_eur_gbp (GBP per EUR — multiply, not divide).


In [ ]:
SHORT_GAP_H = 4   # Maximum gap (hours) filled by linear interpolation

def _wide_to_hourly(wide_df, value_col):
    df = wide_df.rename(columns={'Unnamed: 0': 'date'}).copy()
    df['date'] = pd.to_datetime(df['date'])
    hour_cols = sorted([c for c in df.columns if c.startswith('H') and c[1:].isdigit()],
                       key=lambda x: int(x[1:]))
    melted = df.melt(id_vars='date', value_vars=hour_cols,
                     var_name='hour_col', value_name=value_col)
    melted['offset']    = melted['hour_col'].str[1:].astype(int) - 1
    melted['timestamp'] = melted['date'] + pd.to_timedelta(melted['offset'], unit='h')
    return melted.set_index('timestamp')[value_col].sort_index().astype(float)

xl       = pd.ExcelFile(EXCEL_PATH)
gb_wide  = xl.parse('GB_FINAL')
fr_wide  = xl.parse('FR_FINAL')
fx_daily = xl.parse('GBP-EUR')

gb_s = _wide_to_hourly(gb_wide, 'gb_price_gbp')
fr_s = _wide_to_hourly(fr_wide, 'fr_price_eur')

fx = fx_daily.copy()
fx.columns = ['date', 'fx_eur_gbp']
fx['date'] = pd.to_datetime(fx['date'])
fx = fx.dropna(subset=['fx_eur_gbp']).set_index('date').sort_index()
fx = fx.reindex(pd.date_range(fx.index.min(), fx.index.max(), freq='D')).ffill()

print(f'GB:  {len(gb_s):,} hourly obs  '
      f'({gb_s.index.min().date()} to {gb_s.index.max().date()})')
print(f'FR:  {len(fr_s):,} hourly obs  '
      f'({fr_s.index.min().date()} to {fr_s.index.max().date()})')
print(f'FX:  {len(fx):,} daily obs  '
      f'({fx.index.min().date()} to {fx.index.max().date()})')


## Section 3: Data Cleaning & Validation

**Steps:**
1. Align GB and FR to their common date range
2. Reindex to a complete hourly grid — surfaces missing observations
3. Fill short gaps (≤4 hours) by linear interpolation (DST spring-forward, data gaps)
4. Forward-fill FX from business-day to hourly frequency
5. Convert FR prices EUR → GBP; compute spread = GB − FR
6. Aggregate to daily means (spread-direct model operates at daily frequency)


In [ ]:
panel = pd.concat([gb_s, fr_s], axis=1).sort_index()
panel = panel[~panel.index.duplicated(keep='last')]
start = max(panel['gb_price_gbp'].first_valid_index(),
            panel['fr_price_eur'].first_valid_index())
end   = min(panel['gb_price_gbp'].last_valid_index(),
            panel['fr_price_eur'].last_valid_index())
panel = panel.loc[start:end]

panel = panel.reindex(pd.date_range(panel.index.min(), panel.index.max(), freq='h'))
n_missing = panel.isna().any(axis=1).sum()
panel = panel.interpolate(method='linear', limit=SHORT_GAP_H).dropna()

panel['fx_eur_gbp'] = pd.Series(
    panel.index.normalize().map(fx['fx_eur_gbp']), index=panel.index
).bfill().ffill()
panel['fr_price_gbp'] = panel['fr_price_eur'] * panel['fx_eur_gbp']
panel['spread_gbp']   = panel['gb_price_gbp'] - panel['fr_price_gbp']
panel = panel.dropna(subset=['gb_price_gbp', 'fr_price_gbp', 'spread_gbp'])

daily       = panel[['gb_price_gbp', 'fr_price_gbp', 'spread_gbp']].resample('D').mean().dropna()
_sample_end = daily.index.max()

print('Cleaning summary:')
print(f'  Hourly panel: {panel.index.min().date()} to {panel.index.max().date()}'
      f'  ({len(panel):,} obs)')
print(f'  Gaps filled:  {n_missing} hourly periods (interpolated, limit={SHORT_GAP_H}h)')
print(f'  Daily series: {daily.index.min().date()} to {_sample_end.date()}'
      f'  ({len(daily):,} days)')
print()
print('Last 5 days:')
print(daily.tail(5).round(2).to_string())


## Section 4: Descriptive Statistics

We report level statistics and ADF unit-root tests for each price series and the spread.

**ADF null hypothesis:** series has a unit root (non-stationary).
We expect to **reject** H₀ for the spread — stationarity is required for OU estimation and is consistent with arbitrage limits on the GB–France price differential.


In [ ]:
stats_data = {}
for col, label in [('gb_price_gbp', 'GB price (£/MWh)'),
                    ('fr_price_gbp', 'FR price (£/MWh)'),
                    ('spread_gbp',   'Spread GB-FR (£/MWh)')]:
    s = daily[col]
    stats_data[label] = {
        'N':               len(s),
        'Mean':            round(s.mean(), 3),
        'Std Dev':         round(s.std(), 3),
        'Skewness':        round(sp_stats.skew(s), 3),
        'Excess Kurtosis': round(sp_stats.kurtosis(s), 3),
        'Min':             round(s.min(), 2),
        'Max':             round(s.max(), 2),
    }

print('Descriptive Statistics (daily averages):')
print(pd.DataFrame(stats_data).T.to_string())

print()
print(f'  {"Series":<22}  {"ADF stat":>10}  {"p-value":>10}  {"Result"}')
print('  ' + '-' * 65)
for col, label in [('gb_price_gbp', 'GB price'),
                    ('fr_price_gbp', 'FR price'),
                    ('spread_gbp',   'Spread GB-FR')]:
    stat, p, _, _, crit, _ = adfuller(daily[col].dropna(), autolag='AIC')
    verdict = 'Reject H0 (stationary)' if p < 0.05 else 'Fail to reject H0 (unit root)'
    print(f'  {label:<22}  {stat:>10.4f}  {p:>10.4f}  {verdict}')


## Section 5: Exploratory Data Analysis

Three charts motivate the modelling choices:

- **Fig 1** — GB and FR price levels: identifies the 2022 energy crisis and post-crisis normalisation
- **Fig 2** — Daily GB−FR spread: shows the structural positive bias and its time variation
- **Fig 3** — Spread distribution: excess kurtosis and heavy tails signal the need for jump components


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 12))

# Fig 1: Price levels
ax = axes[0]
ax.plot(daily.index, daily['gb_price_gbp'], lw=0.8, color='steelblue',
        label='GB (GBP/MWh)', alpha=0.9)
ax.plot(daily.index, daily['fr_price_gbp'], lw=0.8, color='darkorange',
        label='FR (GBP/MWh)', alpha=0.9)
ax.axvspan(pd.Timestamp('2021-12-01'), pd.Timestamp('2023-06-01'),
           alpha=0.07, color='red', label='Crisis period')
ax.set_ylabel('GBP/MWh')
ax.set_title('Fig 1. Daily GB and FR Electricity Prices (GBP/MWh)', fontweight='bold')
ax.legend(fontsize=9)

# Fig 2: Spread
ax = axes[1]
ax.plot(daily.index, daily['spread_gbp'], lw=0.8, color='#2ca02c', alpha=0.85)
ax.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)
ax.axhline(daily['spread_gbp'].mean(), color='#2ca02c', lw=1.5, ls=':',
           label=f'Mean = {daily["spread_gbp"].mean():.1f} GBP/MWh')
ax.fill_between(daily.index, 0, daily['spread_gbp'],
                where=daily['spread_gbp'] > 0, alpha=0.15, color='#2ca02c')
ax.fill_between(daily.index, 0, daily['spread_gbp'],
                where=daily['spread_gbp'] < 0, alpha=0.15, color='red')
ax.set_ylabel('GBP/MWh')
ax.set_title('Fig 2. Daily GB-FR Spread  [positive = GB above FR]', fontweight='bold')
ax.legend(fontsize=9)

# Fig 3: Distribution
ax = axes[2]
sv = daily['spread_gbp'].dropna()
ax.hist(sv, bins=80, density=True, color='steelblue', alpha=0.6, label='Observed')
mu_s, sd_s = sv.mean(), sv.std()
xr = np.linspace(sv.min(), sv.max(), 300)
ax.plot(xr, sp_stats.norm.pdf(xr, mu_s, sd_s), 'r-', lw=2,
        label=f'N({mu_s:.1f}, {sd_s:.1f})')
ax.axvline(0, color='k', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Spread (GBP/MWh)')
ax.set_title(
    f'Fig 3. Spread Distribution  '
    f'[skew={sp_stats.skew(sv):.2f}, excess kurtosis={sp_stats.kurtosis(sv):.2f}]',
    fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_eda.png')


## Section 6: Model Specification

### 6.1 Decomposition

Following Cartea & González-Pedraz (2012), the daily spread decomposes as:

$$S_t = f_S(t) + X_t$$

where $f_S(t)$ is the deterministic seasonal baseline (OLS) and $X_t$ is the stochastic deviation (OU + jumps).

### 6.2 Deterministic Component

$$f_S(t) = \beta_0 + \beta_1\tau + \beta_2\sin(2\pi\tau) + \beta_3\cos(2\pi\tau)
           + \beta_4\sin(4\pi\tau) + \beta_5\cos(4\pi\tau) + \beta_6 D_t$$

- $\tau$ = years elapsed since data start (Dec 2021)
- Annual harmonics: $\sin(2\pi\tau)$, $\cos(2\pi\tau)$
- Semi-annual harmonics: $\sin(4\pi\tau)$, $\cos(4\pi\tau)$ — **tested for significance, may be dropped**
- $D_t = 1$ on Saturday/Sunday

**Tau anchoring:** $\tau$ is frozen at 1 January 2025 for projection. This prevents \
extrapolating the crisis-driven upward trend ($\hat\beta_1 = +9.43$ £/MWh/yr) over 25 years.

### 6.3 Stochastic Component — OU with Jumps

$$X_{t+1} = c + \phi X_t + \sigma_d\,\varepsilon_t + J_t^+ - J_t^-$$

- $\phi = e^{-\kappa\Delta t}$, AR(1) coefficient; $\kappa$ = mean-reversion speed (yr⁻¹)
- $J_t^\pm$ = Poisson-exponential jump components (separate positive/negative)

### 6.4 Revenue Formula

$$R_{\text{year}} = \sum_{d=1}^{365} \bigl|S_d\bigr| \times C \times 24 \times A \times \rho \;/\; 10^6 \quad \text{(£m nominal)}$$

where $C = 1000$ MW, $A = 0.9659$, $\rho$ = capture ratio (32.3%).

The **capture ratio** converts gross theoretical revenue to the Ofgem-reported net figure \
(after auction costs, TSO charges, balancing costs).

## Section 7: Full OLS Estimation — All Fourier Harmonics

We estimate the 7-parameter unrestricted model (annual + semi-annual harmonics). This is the basis for all subsequent significance tests.


In [ ]:
def build_features_full(idx, t0):
    tau = (idx - t0).total_seconds() / (365.25 * 24 * 3600)
    return pd.DataFrame({
        'const':   1.0,
        'tau':     tau,
        'sin1':    np.sin(2 * np.pi * tau),
        'cos1':    np.cos(2 * np.pi * tau),
        'sin2':    np.sin(4 * np.pi * tau),
        'cos2':    np.cos(4 * np.pi * tau),
        'weekend': (idx.dayofweek >= 5).astype(float),
    }, index=idx)

t0_spread     = daily.index[0]
feat_full     = build_features_full(daily.index, t0_spread)
res_full      = sm.OLS(daily['spread_gbp'].values, feat_full).fit()
f_spread_full = pd.Series(res_full.fittedvalues, index=daily.index)
resid_full    = daily['spread_gbp'] - f_spread_full

n_obs  = len(daily)
k_full = feat_full.shape[1]
rss_full   = float((resid_full ** 2).sum())
sigma_full = float(resid_full.std())

print(f'Full OLS (7 parameters):  R2={res_full.rsquared:.4f}  '
      f'sigma_resid={sigma_full:.4f} GBP/MWh  N={n_obs}')
print()
print(f'  {"Param":<10}  {"Coeff":>10}  {"Std Err":>10}  {"t-stat":>10}  {"p-value":>10}  Sig')
print('  ' + '-' * 68)
for i, param in enumerate(feat_full.columns):
    coef = res_full.params[i]
    se   = res_full.bse[i]
    t    = res_full.tvalues[i]
    p    = res_full.pvalues[i]
    sig  = ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
    print(f'  {param:<10}  {coef:>10.4f}  {se:>10.4f}  {t:>10.4f}  {p:>10.4f}  {sig}')
print('  *** p<0.001  ** p<0.01  * p<0.05')


## Section 8: Parameter Significance Tests

Four complementary tests determine which harmonics to retain:

1. **Individual t-tests** — each coefficient separately (reported above)
2. **Joint F-test** — all Fourier terms together vs trend+weekend only
3. **Partial F-test** — semi-annual terms (sin₂, cos₂) beyond annual terms
4. **AIC/BIC** — penalised model comparison, full vs annual-only
5. **Amplitude analysis** — economic magnitude of each harmonic vs noise floor


In [ ]:
# Test 1: F-test — all Fourier terms jointly (vs trend+weekend only)
res_no_fourier = sm.OLS(
    daily['spread_gbp'].values, feat_full[['const', 'tau', 'weekend']]
).fit()
rss_no_fourier = float((daily['spread_gbp'] - res_no_fourier.fittedvalues).values ** 2 @
                       np.ones(n_obs))
q1 = 4
F1 = ((rss_no_fourier - rss_full) / q1) / (rss_full / (n_obs - k_full))
p1 = sp_stats.f.sf(F1, q1, n_obs - k_full)

# Test 2: Partial F-test — sin2, cos2 only (annual already included)
feat_annual  = feat_full[['const', 'tau', 'sin1', 'cos1', 'weekend']]
res_annual   = sm.OLS(daily['spread_gbp'].values, feat_annual).fit()
rss_annual   = float((daily['spread_gbp'] - res_annual.fittedvalues).values ** 2 @
                     np.ones(n_obs))
k_annual     = feat_annual.shape[1]
q2 = 2
F2 = ((rss_annual - rss_full) / q2) / (rss_full / (n_obs - k_full))
p2 = sp_stats.f.sf(F2, q2, n_obs - k_full)

# Test 3: AIC / BIC
def ols_aic_bic(rss, n, k):
    log_lik = -n / 2 * (1 + np.log(2 * np.pi * rss / n))
    return -2 * log_lik + 2 * k, -2 * log_lik + k * np.log(n)

aic_full,   bic_full   = ols_aic_bic(rss_full,   n_obs, k_full)
aic_annual, bic_annual = ols_aic_bic(rss_annual, n_obs, k_annual)

# Test 4: Amplitudes
pf = pd.Series(res_full.params, index=feat_full.columns)
amp_ann  = np.sqrt(pf['sin1']**2 + pf['cos1']**2)
amp_semi = np.sqrt(pf['sin2']**2 + pf['cos2']**2)

# Results
print('=' * 65)
print('  TEST 1 — Joint F-test: All Fourier terms (sin1, cos1, sin2, cos2)')
print(f'  F({q1}, {n_obs-k_full}) = {F1:.3f}   p = {p1:.4f}')
verdict1 = 'REJECT H0 — seasonal variation exists' if p1 < 0.05 else 'Fail to reject H0'
print(f'  => {verdict1}')

print()
print('  TEST 2 — Partial F-test: Semi-annual only (sin2, cos2)')
print(f'  F({q2}, {n_obs-k_full}) = {F2:.3f}   p = {p2:.4f}')
verdict2 = ('Significant at 5%' if p2 < 0.05
            else 'NOT significant — sin2/cos2 add little beyond annual terms')
print(f'  => {verdict2}')

print()
print('  TEST 3 — AIC/BIC Comparison')
print(f'  {"Model":<25}  {"k":>3}  {"AIC":>12}  {"BIC":>12}')
print(f'  {"-"*57}')
print(f'  {"Full (7 params)":<25}  {k_full:>3}  {aic_full:>12.1f}  {bic_full:>12.1f}')
print(f'  {"Annual only (5 params)":<25}  {k_annual:>3}  {aic_annual:>12.1f}  {bic_annual:>12.1f}')
print(f'  delta-AIC = {aic_annual - aic_full:+.1f}    delta-BIC = {bic_annual - bic_full:+.1f}')
bic_pref = 'Annual-only' if bic_annual < bic_full else 'Full'
print(f'  BIC prefers: {bic_pref} model')

print()
print('  TEST 4 — Seasonal Amplitude vs Noise Floor')
print(f'  Annual amplitude  (sin1, cos1): {amp_ann:.2f} GBP/MWh')
print(f'  Semi-ann amplitude (sin2, cos2): {amp_semi:.2f} GBP/MWh')
print(f'  sigma_resid (noise):            {sigma_full:.2f} GBP/MWh')
print(f'  Annual / sigma:     {amp_ann/sigma_full:.3f}')
print(f'  Semi-annual / sigma:{amp_semi/sigma_full:.3f}')

print()
print('  CONCLUSION')
print('  ' + '-' * 60)
print(f'  Fourier terms are jointly significant (F={F1:.0f}, p<0.001)')
print(f'  Semi-annual terms are NOT significant (F={F2:.3f}, p={p2:.3f})')
print(f'  BIC favours the annual-only model (delta-BIC={bic_annual-bic_full:+.1f})')
print(f'  Semi-annual amplitude ({amp_semi:.1f} GBP/MWh) << sigma ({sigma_full:.1f})')
print(f'  GB-FR seasonality reflects a gas/nuclear cost differential')
print(f'  (single annual cycle). Spain-France multi-harmonic seasonality')
print(f'  reflects hydro, solar, and temperature cycles absent here.')
print(f'  => Drop sin2 and cos2. Proceed with annual-only model.')
print('=' * 65)


## Section 9: Model Refinement — Annual-Only OLS

Based on the significance tests, we drop the semi-annual harmonics and re-estimate the \
5-parameter model. This is the **final OLS specification** used in all downstream steps.

$$f_S(t) = \hat\beta_0 + \hat\beta_1\tau + \hat\beta_2\sin(2\pi\tau) + \hat\beta_3\cos(2\pi\tau) + \hat\beta_4 D_t$$

In [ ]:
def build_features_spread(idx, t0):
    tau = (idx - t0).total_seconds() / (365.25 * 24 * 3600)
    return pd.DataFrame({
        'const':   1.0,
        'tau':     tau,
        'sin1':    np.sin(2 * np.pi * tau),
        'cos1':    np.cos(2 * np.pi * tau),
        'weekend': (idx.dayofweek >= 5).astype(float),
    }, index=idx)

feat_spread = build_features_spread(daily.index, t0_spread)
res_ols     = sm.OLS(daily['spread_gbp'].values, feat_spread).fit()
f_spread    = pd.Series(res_ols.fittedvalues, index=daily.index)
resid_s     = daily['spread_gbp'] - f_spread
params_s    = pd.Series(res_ols.params, index=feat_spread.columns)

print(f'Refined OLS (5 parameters):  R2={res_ols.rsquared:.4f}  '
      f'sigma_resid={resid_s.std():.4f} GBP/MWh  N={len(daily)}')
print()
print(f'  {"Param":<10}  {"Coeff":>10}  {"Std Err":>10}  {"t-stat":>10}  {"p-value":>10}  Sig')
print('  ' + '-' * 68)
for i, param in enumerate(feat_spread.columns):
    coef = res_ols.params[i]
    se   = res_ols.bse[i]
    t    = res_ols.tvalues[i]
    p    = res_ols.pvalues[i]
    sig  = ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
    print(f'  {param:<10}  {coef:>10.4f}  {se:>10.4f}  {t:>10.4f}  {p:>10.4f}  {sig}')

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(daily.index, daily['spread_gbp'], lw=0.7, alpha=0.7,
             color='steelblue', label='Actual spread')
axes[0].plot(daily.index, f_spread, lw=1.5, color='darkorange', label='OLS f_S(t)')
axes[0].axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
axes[0].set_ylabel('GBP/MWh'); axes[0].legend()
axes[0].set_title('Fig S1a. Spread OLS Fit — Annual-Only Model', fontweight='bold')

axes[1].plot(daily.index, resid_s, lw=0.7, color='#2ca02c', alpha=0.8)
axes[1].axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
for s in [1, 2, 3]:
    axes[1].axhline(+s * resid_s.std(), color='red', lw=0.5, ls=':', alpha=0.5)
    axes[1].axhline(-s * resid_s.std(), color='red', lw=0.5, ls=':', alpha=0.5)
axes[1].set_ylabel('Residual (GBP/MWh)')
axes[1].set_title('Fig S1b. OLS Residuals  (dotted = +-1/2/3 sigma)', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figS1_spread_ols.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figS1_spread_ols.png')


## Section 10: Jump Detection (Cartea & González-Pedraz)

OLS residuals $\varepsilon_t = S_t - f_S(t)$ contain extreme observations inconsistent with Gaussian noise — **price jumps** from supply/demand imbalances or grid events.

**Iterative ±3σ filter (Cartea & González-Pedraz, 2012):**
1. Flag observations more than 3σ from the mean
2. Remove flagged values and re-estimate σ from the cleaned series
3. Repeat until convergence (≤20 iterations)

Positive jumps (GB spike) and negative jumps (FR spike) are fitted separately with Poisson intensities $\lambda^+$, $\lambda^-$ and exponential size parameters $\beta^+$, $\beta^-$, with quarterly variation in $\lambda$.


In [ ]:
def cartea_filter(resid, label, thresh=3.0):
    clean = resid.copy()
    for _ in range(20):
        mu, sig = clean.mean(), clean.std()
        is_jump = (clean - mu).abs() > thresh * sig
        if not is_jump.any():
            break
        clean[is_jump] = np.nan
    jumps = resid[resid.index.isin(clean[clean.isna()].index)]
    clean = clean.dropna()
    print(f'  {label}: {len(jumps)} jumps / {len(resid)} days '
          f'({len(jumps)/len(resid)*100:.1f}%)   sigma_clean={clean.std():.4f} GBP/MWh')
    return clean, jumps

def build_jump_params(jump_sizes, all_days):
    pos_j = jump_sizes[jump_sizes > 0]
    neg_j = jump_sizes[jump_sizes < 0].abs()
    result = {}
    for sign, jmp in [('pos', pos_j), ('neg', neg_j)]:
        if len(jmp) == 0:
            result[sign] = {'lam_by_q': {q: 0.0 for q in [1,2,3,4]}, 'beta': 0.0, 'n': 0}
            continue
        beta  = float(jmp.mean())
        lam_q = {}
        for q in [1, 2, 3, 4]:
            q_days  = all_days[pd.DatetimeIndex(all_days).quarter == q]
            q_jumps = jmp.index[pd.DatetimeIndex(jmp.index).quarter == q]
            lam_q[q] = len(q_jumps) / max(len(q_days), 1)
        result[sign] = {'lam_by_q': lam_q, 'beta': beta, 'n': len(jmp)}
    return result

print('Jump detection on spread residuals:')
resid_s_clean, jumps_s = cartea_filter(resid_s, 'GB-FR Spread')
jump_params_s = build_jump_params(jumps_s, daily.index)

print()
print(f'  {"Direction":<18}  {"n":>5}  {"Mean lambda/day":>17}  {"Beta (GBP/MWh)":>16}')
print('  ' + '-' * 62)
for sign, lbl in [('pos', 'Positive (GB spike)'), ('neg', 'Negative (FR spike)')]:
    jp = jump_params_s[sign]
    ml = sum(jp['lam_by_q'].values()) / 4
    print(f'  {lbl:<18}  {jp["n"]:>5}  {ml:>17.5f}  {jp["beta"]:>16.2f}')

print()
print('  Quarterly lambda (per day):')
print(f'  {"Q":<5}  {"Positive":>12}  {"Negative":>12}')
for q in [1, 2, 3, 4]:
    lp = jump_params_s['pos']['lam_by_q'][q]
    ln = jump_params_s['neg']['lam_by_q'][q]
    print(f'  Q{q:<4}  {lp:>12.5f}  {ln:>12.5f}')


## Section 11: Ornstein-Uhlenbeck Parameter Estimation

The jump-cleaned residual series $\tilde{X}_t$ is fitted as a discrete-time AR(1):

$$\tilde{X}_{t+1} = c + \phi\tilde{X}_t + \eta_t, \quad \eta_t \sim N(0, \sigma_d^2)$$

Continuous-time parameters: $\kappa = -\ln(\phi) \times 365.25$ yr⁻¹, $\theta = c/(1-\phi)$.

With $\kappa \approx 168$/yr, the half-life of a deviation is $\ln 2 / \kappa \approx 1.5$ days — \
consistent with fast arbitrage dynamics on a liquid interconnector.

In [ ]:
def ou_ar1_levels(series, label):
    y   = series.values
    X   = sm.add_constant(y[:-1])
    res = sm.OLS(y[1:], X).fit()
    c, phi  = float(res.params[0]), float(res.params[1])
    eps     = pd.Series(res.resid, index=series.index[1:])
    sigma_d = float(eps.std())
    kappa   = -np.log(max(phi, 1e-9)) * 365.25
    theta   = c / (1 - phi) if abs(1 - phi) > 1e-9 else 0.0
    hl      = np.log(2) / (kappa / 365.25)
    print(f'  {label}:')
    print(f'    phi={phi:.5f}   kappa={kappa:.2f}/yr   half-life={hl:.1f} days')
    print(f'    theta={theta:.4f} GBP/MWh   sigma_d={sigma_d:.4f} GBP/MWh')
    return {'phi': phi, 'c': c, 'sigma_d': sigma_d,
            'kappa_yr': kappa, 'theta': theta, 'eps': eps}

print('OU estimation (AR(1) on jump-cleaned residuals):')
ou_s = ou_ar1_levels(resid_s_clean, 'GB-FR Spread')

eps = ou_s['eps']
print()
print('OU innovation diagnostics:')
print(f'  Mean:            {eps.mean():.5f}  (approx 0)')
print(f'  Std:             {eps.std():.4f} GBP/MWh')
print(f'  Skewness:        {sp_stats.skew(eps):.4f}')
print(f'  Excess kurtosis: {sp_stats.kurtosis(eps):.4f}')
_, p_norm = sp_stats.normaltest(eps)
print(f'  Normality test:  p={p_norm:.4f}  '
      f'({"reject normality" if p_norm < 0.05 else "consistent with normality"})')


## Section 12: Monte Carlo Simulation — Setup

### Projection dates
Revenue simulated for 25 years from FID (2028–2052), daily Euler scheme.

### Tau anchoring
The OLS trend $\hat\beta_1 \cdot \tau$ is frozen at **1 January 2025**. Without anchoring, f_S would extrapolate a post-crisis recovery trend (+£9.43/MWh/yr → £302/MWh spread by 2052 — clearly implausible).

### Capture ratio
$$\rho = \frac{\text{Actual mean revenue (2024–2025)}}{\text{Theoretical mean revenue (2024–2025)}}$$

2024–2025 only: 2023 is excluded (anomalous — FR nuclear fleet recovery drove above-normal spreads). 2024–2025 is the normalised post-crisis steady state.


In [ ]:
proj_dates = pd.date_range(
    start=f'{FID_YEAR}-01-01',
    end=f'{FID_YEAR + REGIME_YEARS - 1}-12-31',
    freq='D'
)
proj_years = proj_dates.year.values

_tau_anchor_s = float(
    (ANCHOR_DATE - t0_spread).total_seconds()
) / (365.25 * 24 * 3600)

def project_f_spread(params_s, proj_dates, t0, tau_anchor, tau_growth=PROJ_TAU_GROWTH):
    feat = build_features_spread(proj_dates, t0)
    yrs_from_anchor = np.array(
        (proj_dates - ANCHOR_DATE).total_seconds(), dtype=float
    ) / (365.25 * 24 * 3600)
    feat = feat.copy()
    feat['tau'] = tau_anchor + tau_growth * np.clip(yrs_from_anchor, 0, None)
    return feat.values @ params_s.values

f_s_proj = project_f_spread(params_s, proj_dates, t0_spread, _tau_anchor_s)
print(f'Projected f_S(t):')
print(f'  Year 1  (2028) mean: {f_s_proj[:365].mean():.2f} GBP/MWh')
print(f'  Year 25 (2052) mean: {f_s_proj[-365:].mean():.2f} GBP/MWh')

actual_norm_mean = np.mean([v for k, v in ACTUALS.items() if k >= 2024])
spread_2024_25   = daily.loc['2024':'2025', 'spread_gbp'].abs().mean()
theoretical_norm = spread_2024_25 * CAPACITY_MW * 24 * 365 * AVAILABILITY / 1e6
capture_ratio    = actual_norm_mean / theoretical_norm

print()
print(f'Capture ratio (2024-2025 normalised):')
print(f'  Mean |spread| 2024-2025:  {spread_2024_25:.2f} GBP/MWh')
print(f'  Theoretical annual rev:   GBP{theoretical_norm:.1f}m')
print(f'  Actual mean 2024-2025:    GBP{actual_norm_mean:.1f}m (2024: 107.5, 2025: 109.4)')
print(f'  Capture ratio:            {capture_ratio:.4f}  ({capture_ratio*100:.1f}%)')
print(f'  Market-cost deductions:   {(1-capture_ratio)*100:.0f}% '
      f'(auction costs, TSO charges, balancing)')


## Section 13: Monte Carlo Simulation

**Euler scheme:** 10,000 paths × 25 years, daily steps.
**Spread path:** $S_d = f_S(t_d) + X_d$
**Revenue (daily):** $|S_d| \times 1000\,\text{MW} \times 24\,\text{h} \times 0.9659 \times \rho / 10^6$
**Batches:** 500 paths per batch (memory efficiency). Intermediate P50 printed every 2,500 paths.


In [ ]:
rng     = np.random.default_rng(RANDOM_SEED)
BATCH   = 500
n_days  = len(proj_dates)
phi     = ou_s['phi'];   c_ou = ou_s['c'];  sigma_d = ou_s['sigma_d']
jp_pos  = jump_params_s['pos'];  jp_neg = jump_params_s['neg']

annual_rev = np.zeros((N_PATHS, REGIME_YEARS))

print(f'Running {N_PATHS:,} paths x {REGIME_YEARS} years ({n_days:,} days)...')
for b0 in range(0, N_PATHS, BATCH):
    b1 = min(b0 + BATCH, N_PATHS);  n = b1 - b0
    X  = np.zeros(n)
    batch_rev = np.zeros((REGIME_YEARS, n))
    for d in range(n_days):
        X = c_ou + phi * X + sigma_d * rng.standard_normal(n)
        q = proj_dates[d].quarter
        for jp, sign in [(jp_pos, +1.0), (jp_neg, -1.0)]:
            lam = jp['lam_by_q'][q]
            if lam > 0 and jp['beta'] > 0:
                X += sign * (rng.random(n) < lam) * rng.exponential(jp['beta'], n)
        rev_d  = np.abs(f_s_proj[d] + X) * CAPACITY_MW * 24.0 * AVAILABILITY / 1e6 * capture_ratio
        yr_idx = proj_years[d] - FID_YEAR
        if 0 <= yr_idx < REGIME_YEARS:
            batch_rev[yr_idx] += rev_d
    annual_rev[b0:b1] = batch_rev.T
    if (b1 % 2500 == 0) or b1 == N_PATHS:
        print(f'  {b1:>6}/{N_PATHS} paths  =>  Yr1 P50 = '
              f'GBP{np.percentile(annual_rev[:b1, 0], 50):.0f}m')

print('Simulation complete.')


## Section 14: Results — Revenue Tables and Charts

### P10/P50/P90 convention (statistical)
P10 = 10th percentile = downside (low revenue); P90 = 90th percentile = upside (high revenue). P10 < P50 < P90.

### Why revenues are flat across 25 years
The OU process ($\kappa$ = 168/yr, half-life ≈ 1.5 days) converges to its stationary distribution within days — the same distribution applies in every simulation year. The deterministic component $f_S(t)$ is anchored flat. Time variation requires either (a) PROJ_TAU_GROWTH > 0, or (b) cannibalisation adjustment once Arup FA/MA figures for IFA2 are available (scenario analysis, not base case).


In [ ]:
p10 = np.percentile(annual_rev, 10, axis=0)
p50 = np.percentile(annual_rev, 50, axis=0)
p90 = np.percentile(annual_rev, 90, axis=0)

rev_df = pd.DataFrame({
    'cal_year':  [FID_YEAR + i for i in range(REGIME_YEARS)],
    'regime_yr': range(1, REGIME_YEARS + 1),
    'p10_gbpm':  p10.round(2), 'p50_gbpm': p50.round(2), 'p90_gbpm': p90.round(2),
})
rev_df.to_csv(OUTPUT_DIR / 'mc_revenue_spread_direct.csv', index=False)
print('Saved: mc_revenue_spread_direct.csv')
print()
print(f'  {"Yr":>3}  {"Cal":>6}  {"P10 (GBPm)":>12}  {"P50 (GBPm)":>12}  {"P90 (GBPm)":>12}')
print('  ' + '-' * 50)
rows = list(range(5)) + list(range(5, REGIME_YEARS, 5)) + [REGIME_YEARS-1]
for yr in sorted(set(rows)):
    print(f'  {yr+1:>3}  {FID_YEAR+yr:>6}  {p10[yr]:>12.1f}  {p50[yr]:>12.1f}  {p90[yr]:>12.1f}')

cal_years = [FID_YEAR + i for i in range(REGIME_YEARS)]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.fill_between(cal_years, p10, p90, alpha=0.25, color='steelblue', label='P10-P90 band')
ax.plot(cal_years, p50, lw=2.0, color='steelblue', label='P50 (median)')
ax.plot(cal_years, p10, lw=0.9, ls='--', color='steelblue')
ax.plot(cal_years, p90, lw=0.9, ls='--', color='steelblue')
for yr, val in ACTUALS.items():
    ax.scatter([yr], [val], color='red', zorder=5, s=50)
ax.scatter([], [], color='red', s=50, label='Ofgem actuals 2022-25')
ax.set_title('M2: Spread-Direct OU\n(FA base case, capture ratio applied)',
             fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('GBPm nominal'); ax.legend(fontsize=9)

ax2 = axes[1]
try:
    m1 = pd.read_csv(IC_DIR / 'M1' / 'mc_revenue_projections.csv')
    ax2.fill_between(m1['cal_year'], m1['p10_gbpm'], m1['p90_gbpm'],
                     alpha=0.12, color='darkorange')
    ax2.plot(m1['cal_year'], m1['p50_gbpm'], lw=2, color='darkorange',
             label='M1 P50 (log-price OU)')
    ax2.plot(m1['cal_year'], m1['p10_gbpm'], lw=0.9, ls='--', color='darkorange')
    ax2.plot(m1['cal_year'], m1['p90_gbpm'], lw=0.9, ls='--', color='darkorange')
except FileNotFoundError:
    ax2.text(0.5, 0.5, 'M1 results not found', transform=ax2.transAxes, ha='center')
ax2.fill_between(cal_years, p10, p90, alpha=0.20, color='steelblue')
ax2.plot(cal_years, p50, lw=2, color='steelblue', label='M2 P50 (spread-direct OU)')
ax2.plot(cal_years, p10, lw=0.9, ls='--', color='steelblue')
ax2.plot(cal_years, p90, lw=0.9, ls='--', color='steelblue')
for yr, val in ACTUALS.items():
    ax2.scatter([yr], [val], color='red', zorder=5, s=50)
ax2.scatter([], [], color='red', s=50, label='Ofgem actuals 2022-25')
ax2.set_title('Methodology Comparison — M1 (orange) vs M2 (blue)', fontweight='bold')
ax2.set_xlabel('Year'); ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figS2_methodology_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figS2_methodology_comparison.png')


## Section 15: Export to Ofgem Cap & Floor Model

Revenue projections are written into the Ofgem Excel Cap & Floor template (IFA2 FPA). Three copies — P10, P50, P90 — are produced. Values go to `Input` sheet, row 25, columns V–AT (columns 22–46, regime years 2028–2052).

> **Note:** Verify cap/floor parameters in the template against the actual Ofgem Window 2 IFA2 decision document before citing any figures.


In [ ]:
import shutil
import openpyxl

EXCEL_TEMPLATE = IC_DIR / 'copy_cap_and_floor_financial_model_-_ifa2_fpa.xlsm'
INPUT_ROW, COL_YEAR_1 = 25, 22

for pct_label, values in [('p10', p10), ('p50', p50), ('p90', p90)]:
    out_path = OUTPUT_DIR / f'ic_cap_floor_m2_{pct_label}.xlsm'
    shutil.copy2(EXCEL_TEMPLATE, out_path)
    wb = openpyxl.load_workbook(out_path, keep_vba=True)
    ws = wb['Input']
    for i, val in enumerate(values):
        ws.cell(row=INPUT_ROW, column=COL_YEAR_1 + i, value=round(float(val), 2))
    wb.save(out_path)
    wb.close()
    print(f'Written: {out_path.name}  (Yr1=GBP{values[0]:.1f}m, Yr25=GBP{values[-1]:.1f}m)')

print()
print('Open ic_cap_floor_m2_*.xlsm in Excel to evaluate cap/floor breach.')
